<a href="https://colab.research.google.com/github/serahnjogu-new/Climate-and-health-risk-prediction/blob/main/Top_performing_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import warnings
from scipy.optimize import minimize
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import KMeans
from sklearn.metrics import f1_score, roc_auc_score

warnings.filterwarnings('ignore')

# 1. Load Data
print("Loading datasets...")
train   = pd.read_csv('/content/Train (6).csv')
test    = pd.read_csv('/content/Test (7).csv')
climate = pd.read_csv('/content/climate_features.csv')
ss      = pd.read_csv('/content/SampleSubmission (5).csv')

train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 2. Targeted Elderly Vulnerability Feature Engineering
def engineer_features_elderly_focus(train_df, test_df):
    combined = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

    kmeans = KMeans(n_clusters=12, random_state=42).fit(combined[['latitude', 'longitude']])
    combined['geo_cluster'] = kmeans.predict(combined[['latitude', 'longitude']])

    train_feat = combined.iloc[:len(train_df)].copy()
    test_feat  = combined.iloc[len(train_df):].copy()

    def process(df):
        df = df.copy()
        df['deathdate'] = pd.to_datetime(df['deathdate'])
        df['year']  = df['deathdate'].dt.year
        df['month'] = df['deathdate'].dt.month

        # Core features
        df['log_age']       = np.log1p(df['age'])
        df['is_infant']     = (df['age'] <= 2).astype(int)
        df['is_elderly']    = (df['age'] >= 60).astype(int)

        # Explicit Age Brackets to catch non-linear boundaries cleanly
        df['age_bracket']   = pd.cut(df['age'], bins=[-1, 2, 18, 40, 60, 120], labels=[0, 1, 2, 3, 4]).astype(int)

        df['temp_range']    = df['max_temperature'] - df['min_temperature']
        df['heat_stress']   = df['hot_days_30d'] * df['max_temperature']
        df['drought_idx']   = df['rain_sum_30d'] / (df['ndvi_30d'] + 1e-5)
        df['temp_ndvi']     = df['max_temperature'] / (df['ndvi_30d'] + 1e-5)

        # TARGETED ELDERLY STRESS INTERACTIONS (Fixing the False Negative Blind Spot)
        df['elderly_heat_stress'] = df['is_elderly'] * df['heat_stress']
        df['elderly_max_temp']    = df['is_elderly'] * df['max_temperature']
        df['elderly_drought']     = df['is_elderly'] * df['drought_idx']

        year_sensitivity = {
            2007: 1.000, 2008: 0.796, 2009: 0.770, 2010: 0.769, 2011: 0.658, 2012: 0.703,
            2013: 0.563, 2014: 0.688, 2015: 0.615, 2016: 0.628, 2017: 0.573, 2018: 0.542,
            2019: 0.459, 2020: 0.533, 2021: 0.537, 2022: 0.661
        }
        df['year_sensitivity'] = df['year'].map(year_sensitivity)

        df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
        df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

        drop_cols = ['ID', 'deathdate', 'location', 'deathdate_extra']
        return df.drop(drop_cols, axis=1, errors='ignore')

    return process(train_feat), process(test_feat)

train_df, test_df = engineer_features_elderly_focus(train, test)

X = train_df.drop('is_climate_sensitive', axis=1).fillna(-999)
y = train_df['is_climate_sensitive']
X_test = test_df.drop('is_climate_sensitive', axis=1, errors='ignore').fillna(-999)

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
neg, pos = (y==0).sum(), (y==1).sum()
scale_pos_weight_val = neg / pos

oof_lgbm = np.zeros(len(X))
oof_xgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))

test_lgbm = np.zeros(len(X_test))
test_xgb  = np.zeros(len(X_test))
test_cat  = np.zeros(len(X_test))

print("\n--- Training Model with Targeted Elderly Interactions ---")
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    lgbm = LGBMClassifier(
        n_estimators=3000, learning_rate=0.01, max_depth=6, num_leaves=40,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.7,
        subsample=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42
    )
    lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='auc',
             callbacks=[early_stopping(200), log_evaluation(0)])

    xgb = XGBClassifier(
        n_estimators=3000, learning_rate=0.01, max_depth=5,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.6,
        subsample=0.8, gamma=0.1, reg_alpha=0.2, reg_lambda=1.5,
        random_state=42, early_stopping_rounds=200, eval_metric='auc'
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    cat = CatBoostClassifier(
        iterations=3000, learning_rate=0.015, depth=5,
        scale_pos_weight=scale_pos_weight_val, l2_leaf_reg=3.0,
        random_seed=42, early_stopping_rounds=200, verbose=False
    )
    cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    oof_lgbm[val_idx] = lgbm.predict_proba(X_val)[:, 1]
    oof_xgb[val_idx]  = xgb.predict_proba(X_val)[:, 1]
    oof_cat[val_idx]  = cat.predict_proba(X_val)[:, 1]

    test_lgbm += lgbm.predict_proba(X_test)[:, 1] / folds.n_splits
    test_xgb  += xgb.predict_proba(X_test)[:, 1] / folds.n_splits
    test_cat  += cat.predict_proba(X_test)[:, 1] / folds.n_splits

# 3. Optimized Weight Blending
def objective_function(weights):
    w1, w2, w3 = weights
    blend = (w1 * oof_lgbm) + (w2 * oof_xgb) + (w3 * oof_cat)
    return -roc_auc_score(y, blend)

res = minimize(objective_function, [0.33, 0.33, 0.33], bounds=[(0,1), (0,1), (0,1)], constraints={'type': 'eq', 'fun': lambda w: 1 - sum(w)})
w_lgbm, w_xgb, w_cat = res.x

oof_ensemble = (w_lgbm * oof_lgbm) + (w_xgb * oof_xgb) + (w_cat * oof_cat)
test_ensemble = (w_lgbm * test_lgbm) + (w_xgb * test_xgb) + (w_cat * test_cat)

# 4. Threshold Optimization
best_thresh, best_score = 0.5, 0
for thresh in np.arange(0.1, 0.9, 0.001):
    f1 = f1_score(y, (oof_ensemble >= thresh).astype(int))
    auc = roc_auc_score(y, oof_ensemble)
    score = (0.6 * f1) + (0.4 * auc)
    if score > best_score:
        best_score, best_thresh = score, thresh

print(f"\n--- Elderly-Focused Model Results ---")
print(f"Best Decision Threshold: {best_thresh:.3f}")
print(f"Final OOF Competition Score: {best_score:.5f}")
print(f"Final OOF ROC-AUC: {roc_auc_score(y, oof_ensemble):.5f}")

# 5. Save Submission
sub = ss.copy()
sub['TargetF1'] = (test_ensemble >= best_thresh).astype(int)
sub['TargetRAUC'] = test_ensemble
sub_filename = '/content/submission_elderly_focused.csv'
sub.to_csv(sub_filename, index=False)
print(f"\nSaved '{sub_filename}' successfully!")

from google.colab import files
files.download(sub_filename)

Loading datasets...

--- Training Model with Targeted Elderly Interactions ---
[LightGBM] [Info] Number of positive: 1637, number of negative: 879
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004007 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5613
[LightGBM] [Info] Number of data points in the train set: 2516, number of used features: 37
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.650636 -> initscore=0.621836
[LightGBM] [Info] Start training from score 0.621836
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 200 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import warnings
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import KMeans
from sklearn.metrics import f1_score, roc_auc_score

warnings.filterwarnings('ignore')

# 2. Load Data explicitly from the /content/ directory path
train   = pd.read_csv('/content/Train (6).csv')
test    = pd.read_csv('/content/Test (7).csv')
climate = pd.read_csv('/content/climate_features.csv')
ss      = pd.read_csv('/content/SampleSubmission (5).csv')

# Merge climate features using the 'ID' column
train = train.merge(climate, on='ID', how='left', suffixes=('', '_extra'))
test  = test.merge(climate, on='ID', how='left', suffixes=('', '_extra'))

# 3. High-Performance Geospatial & Climate Feature Engineering
def engineer_features(train_df, test_df):
    # Combine for global spatial clustering
    combined = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

    # K-Means Spatial Clusters based on Latitude & Longitude
    kmeans = KMeans(n_clusters=10, random_state=42).fit(combined[['latitude', 'longitude']])
    combined['geo_cluster'] = kmeans.predict(combined[['latitude', 'longitude']])

    # Split back
    train_feat = combined.iloc[:len(train_df)].copy()
    test_feat  = combined.iloc[len(train_df):].copy()

    def process(df):
        df = df.copy()
        df['deathdate'] = pd.to_datetime(df['deathdate'])
        df['year']  = df['deathdate'].dt.year
        df['month'] = df['deathdate'].dt.month
        df['day']   = df['deathdate'].dt.day
        df['dayofweek'] = df['deathdate'].dt.dayofweek

        # Season mapping
        df['season'] = df['month'].map({
            12:0, 1:0, 2:0, 3:1, 4:1, 5:1,
            6:2, 7:2, 8:2, 9:3, 10:3, 11:3
        })

        # Non-linear Age & Vulnerability features
        df['log_age']      = np.log1p(df['age'])
        df['age_squared']  = df['age'] ** 2
        df['age_cubed']    = df['age'] ** 3
        df['is_infant']    = (df['age'] == 0).astype(int)
        df['is_toddler']   = (df['age'].between(1, 2)).astype(int)
        df['is_child']     = (df['age'].between(1, 5)).astype(int)
        df['is_elderly']   = (df['age'] >= 60).astype(int)

        # Climate & Geospatial Interactions
        df['temp_range']       = df['max_temperature'] - df['min_temperature']
        df['year_avg_temp']    = df.groupby('year')['max_temperature'].transform('mean')
        df['temp_anomaly']     = df['max_temperature'] - df['year_avg_temp']
        df['temp_ndvi_ratio']  = df['max_temperature'] / (df['ndvi_30d'] + 1e-5)
        df['rain_temp_product']= df['precipitation'] * df['tavg_30d']
        df['lat_lon_product']  = df['latitude'] * df['longitude']

        # Acute Weather Stress
        df['heat_stress_index']= df['hot_days_30d'] * df['max_temperature']
        df['drought_index']    = df['rain_sum_30d'] / (df['ndvi_30d'] + 1e-5)
        df['age_heat_interaction'] = df['age'] * df['hot_days_30d']

        # Historical year sensitivity mapping
        year_sensitivity = {
            2007: 1.000, 2008: 0.796, 2009: 0.770, 2010: 0.769, 2011: 0.658, 2012: 0.703,
            2013: 0.563, 2014: 0.688, 2015: 0.615, 2016: 0.628, 2017: 0.573, 2018: 0.542,
            2019: 0.459, 2020: 0.533, 2021: 0.537, 2022: 0.661
        }
        df['year_sensitivity'] = df['year'].map(year_sensitivity)
        df['year_sens_age']    = df['year_sensitivity'] * df['age']

        # Categorical encoding
        df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
        df['zone']   = df['zone'].map({'Rural': 0, 'Peri_urban': 1})

        drop_cols = ['ID', 'deathdate', 'location', 'deathdate_extra', 'year_avg_temp']
        return df.drop(drop_cols, axis=1, errors='ignore')

    return process(train_feat), process(test_feat)

train_df, test_df = engineer_features(train, test)

X = train_df.drop('is_climate_sensitive', axis=1).fillna(-999)
y = train_df['is_climate_sensitive']
X_test = test_df.drop('is_climate_sensitive', axis=1, errors='ignore').fillna(-999)

# 4. Stratified K-Fold Training
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
neg, pos = (y==0).sum(), (y==1).sum()
scale_pos_weight_val = neg / pos

oof_lgbm = np.zeros(len(X))
oof_xgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))

test_lgbm = np.zeros(len(X_test))
test_xgb  = np.zeros(len(X_test))
test_cat  = np.zeros(len(X_test))

print("Training base models with geospatial features...")
for fold, (train_idx, val_idx) in enumerate(folds.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # LightGBM
    lgbm = LGBMClassifier(
        n_estimators=3000, learning_rate=0.018, max_depth=6, num_leaves=35,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.75,
        subsample=0.85, reg_alpha=0.05, reg_lambda=0.8, random_state=42
    )
    lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='auc',
             callbacks=[early_stopping(200), log_evaluation(0)])

    # XGBoost
    xgb = XGBClassifier(
        n_estimators=3000, learning_rate=0.018, max_depth=5,
        scale_pos_weight=scale_pos_weight_val, colsample_bytree=0.75,
        subsample=0.85, gamma=0.05, reg_alpha=0.05, reg_lambda=0.8,
        random_state=42, early_stopping_rounds=200, eval_metric='auc'
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    # CatBoost
    cat = CatBoostClassifier(
        iterations=3000, learning_rate=0.02, depth=6,
        scale_pos_weight=scale_pos_weight_val, l2_leaf_reg=2.5,
        random_seed=42, early_stopping_rounds=200, verbose=False
    )
    cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))

    # OOF Predictions
    oof_lgbm[val_idx] = lgbm.predict_proba(X_val)[:, 1]
    oof_xgb[val_idx]  = xgb.predict_proba(X_val)[:, 1]
    oof_cat[val_idx]  = cat.predict_proba(X_val)[:, 1]

    # Test Predictions
    test_lgbm += lgbm.predict_proba(X_test)[:, 1] / folds.n_splits
    test_xgb  += xgb.predict_proba(X_test)[:, 1] / folds.n_splits
    test_cat  += cat.predict_proba(X_test)[:, 1] / folds.n_splits

# 5. Optimized Weighted Ensemble Blend
oof_ensemble = (0.45 * oof_lgbm) + (0.35 * oof_xgb) + (0.20 * oof_cat)
test_ensemble = (0.45 * test_lgbm) + (0.35 * test_xgb) + (0.20 * test_cat)

# 6. Fine-grained Threshold Optimization
best_thresh, best_score = 0.5, 0
for thresh in np.arange(0.1, 0.9, 0.001):
    f1 = f1_score(y, (oof_ensemble >= thresh).astype(int))
    auc = roc_auc_score(y, oof_ensemble)
    score = (0.6 * f1) + (0.4 * auc)
    if score > best_score:
        best_score, best_thresh = score, thresh

print(f"\n--- Geospatial Enhanced Results ---")
print(f"Best Decision Threshold: {best_thresh:.3f}")
print(f"OOF Competition Score: {best_score:.5f}")
print(f"OOF ROC-AUC: {roc_auc_score(y, oof_ensemble):.5f}")
print(f"OOF F1-Score: {f1_score(y, (oof_ensemble >= best_thresh).astype(int)):.5f}")

# 7. Save Submission File & Trigger Download
sub = ss.copy()
sub['TargetF1'] = (test_ensemble >= best_thresh).astype(int)
sub['TargetRAUC'] = test_ensemble
sub_filename = '/content/submission_geospatial_boost.csv'
sub.to_csv(sub_filename, index=False)
print(f"\nSaved '{sub_filename}' successfully!")

from google.colab import files
files.download(sub_filename)

In [ ]:
import pandas as pd
import numpy as np

# 1. Load your top two submission files
# Make sure the file names match what you have in your Colab environment
sub1 = pd.read_csv('/content/submission_geospatial_boost.csv')        # Your best private leaderboard performer
sub2 = pd.read_csv('/content/submission_elderly_focused.csv')    # Our new demographic-focused model

# 2. Extract probability columns (TargetRAUC holds the raw soft probabilities)
prob1 = sub1['TargetRAUC'].values
prob2 = sub2['TargetRAUC'].values

# 3. Blend probabilities (50/50 blend or adjust weights e.g., 0.6 / 0.4)
blended_probs = (0.6 * prob1) + (0.4 * prob2)

# 4. Use the optimized threshold (~0.43 to 0.48 typically, let's use 0.45 or derive it)
# We can use the same threshold logic from our previous run
best_thresh = 0.45

# 5. Create final blended submission
sub_final = sub1.copy()
sub_final['TargetRAUC'] = blended_probs
sub_final['TargetF1'] = (blended_probs >= best_thresh).astype(int)

final_filename = '/content/submission_final_blended.csv'
sub_final.to_csv(final_filename, index=False)
print(f"Saved '{final_filename}' successfully!")

from google.colab import files
files.download(final_filename)

Saved '/content/submission_final_blended.csv' successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>